In [ ]:
import pandas as pd
import sqlite3

# Прогноз удовлетворённости клиента для маркетплейса Olist

**Автор:** Данияр Баекеев<br>
**Дата:** Сентябрь 2026<br>
**Датасет:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)


---

## Введение

### Контекст
Olist — бразильский маркетплейс, объединяющий множество продавцов на одной платформе. Датасет содержит реальные анонимизированные данные о заказах, товарах, продавцах, оплатах, доставке и отзывах клиентов.

### Проблема
Низкая оценка клиента — это не только негативный отзыв, но и сигнал о проблемах с логистикой, стоимостью доставки, товаром или продавцом. Обычно бизнес узнаёт об этом уже после публикации отзыва.

### Вопрос
Можно ли предсказать, что клиент поставит низкую оценку (1–2 звезды), используя только данные, которые уже известны к моменту доставки заказа?

### Цель
Построить модель бинарной классификации, которая оценивает риск низкой оценки и позволяет заранее выделять заказы, требующие внимания клиентского сервиса.


---

### 1. Подготовка и очистка датасета

In [2]:
customers = pd.read_csv('datasets/olist_customers_dataset.csv')
geolocation = pd.read_csv('datasets/olist_geolocation_dataset.csv')
order_items = pd.read_csv('datasets/olist_order_items_dataset.csv')
order_payments = pd.read_csv('datasets/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('datasets/olist_order_reviews_dataset.csv')
orders = pd.read_csv('datasets/olist_orders_dataset.csv')
products = pd.read_csv('datasets/olist_products_dataset.csv')
sellers = pd.read_csv('datasets/olist_sellers_dataset.csv')
category_translation = pd.read_csv('datasets/product_category_name_translation.csv')

tables = {
    'customers': customers,
    'geolocation': geolocation,
    'order_items': order_items,
    'order_payments': order_payments,
    'order_reviews': order_reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'category_translation': category_translation
}

In [3]:
# Быстрая диагностика пропусков по всем таблицам
diagnostics = []

for table_name, df in tables.items():
    for col in df.columns:
        diagnostics.append({
            'Таблица': table_name,
            'Признак': col,
            'Тип данных': str(df[col].dtype),
            'Пропуски': int(df[col].isna().sum()),
            'Пропуски (%)': round(df[col].isna().mean() * 100, 2),
            'Уникальных значений': int(df[col].nunique(dropna=False))
        })

report_df = pd.DataFrame(diagnostics)

print('Топ-20 признаков по доле пропусков:')
display(
    report_df
    .sort_values('Пропуски (%)', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

Топ-20 признаков по доле пропусков:


,Таблица,Признак,Тип данных,Пропуски,Пропуски (%),Уникальных значений
0,order_reviews,review_comment_title,str,87656,88.34,4528
1,order_reviews,review_comment_message,str,58247,58.70,36160
2,orders,order_delivered_customer_date,str,2965,2.98,95665
3,products,product_name_lenght,float64,610,1.85,67
4,products,product_photos_qty,float64,610,1.85,20
5,products,product_category_name,str,610,1.85,74
6,products,product_description_lenght,float64,610,1.85,2961
7,orders,order_delivered_carrier_date,str,1783,1.79,81019
8,orders,order_approved_at,str,160,0.16,90734
9,products,product_weight_g,float64,2,0.01,2205
